In [20]:
from SEDataObjects import *
from SEDataObjects.BaseLayer import *
from SEDataObjects.transitWrappers import *
from SEDataObjects.constants import GEODESIC_CRS
import geopandas as gpd
import folium

In [21]:
CONFIG_MAP_AREA_PATH = "map_area_path"
CONFIG_MAP_AREA_PROJECTED_CRS = "map_area_projected_crs"
CONFIG_GTFS_TRANSITLAND_URL = "gtfs_url"
CONFIG_GTFS_TRANSITLAND_KEY_PATH = "gtfs_key_path"
CONFIG_GTFS_CACHE_FOLDER = "gtfs_cache_folder"
CONFIG_FTA_FACILITY_INVENTORY_PATH = "fta_facility_inventory_path"
CONFIG_CITYBIKES_URL = "citybikes_url"
CONFIG_AFDC_API_KEY = "afdc_api_key"
CONFIG_AFDC_URL = "afdc_url"
CONFIG_OSM_CACHE_FOLDER = "osm_cache_folder"
CONFIG_EPA_EJSCREEN_PATH = "epa_ejscreen_path"
CONFIG_SMART_LOCATION_PATH = "smart_location_path"

# COMPLETE CONFIG HERE
CONFIG = {
    CONFIG_MAP_AREA_PATH: "./rawData/santacruz_county.geojson", # The path to the map area spatial data file
    CONFIG_MAP_AREA_PROJECTED_CRS: 26971, # The EPSG number for a projected crs with units in meters covering the map area
    CONFIG_GTFS_TRANSITLAND_URL: "https://transit.land/api/v2/rest/feeds.json", # A path to the Transitland V2 api. See https://www.interline.io/transitland/plans-pricing/
    CONFIG_GTFS_TRANSITLAND_KEY_PATH: "./rawData/TRANSITLAND_KEY", # A path to a file contaiing your Transitland API key
    CONFIG_GTFS_CACHE_FOLDER: "./cache/gtfs_cache/santacruz_county", # A path to a folder where GTFS-static feeds and extracted data canbe stored. The folder must exist, and is recommended to be in cache/gtfs_cache/**
    CONFIG_FTA_FACILITY_INVENTORY_PATH: "./rawData/2023 Facility Inventory.xlsx", # A path to the FTA facility inventory sheet
    CONFIG_CITYBIKES_URL: "http://api.citybik.es/", # The Citybikes API endpoint. See https://citybik.es
    CONFIG_OSM_CACHE_FOLDER: "./cache/osmnx_cache", # A folder to use as the OSMNX cache
    CONFIG_AFDC_URL: "https://developer.nrel.gov/api/alt-fuel-stations/v1/nearest.geojson", # A url to the NREL AFDC alternate fuel stations nearby api endpoint
    CONFIG_AFDC_API_KEY: "./rawData/AFDC_API_KEY", # A path to a folder with a NREL api key. See https://developer.nrel.gov/docs/api-key/
    CONFIG_EPA_EJSCREEN_PATH: "rawData/EJScreen_2024_BG_with_AS_CNMI_GU_VI.gdb", # A path to the Ejscreen gdb
    CONFIG_SMART_LOCATION_PATH: "rawData/SmartLocationDatabaseV3/SmartLocationDatabase.gdb", # A path to the Smart Location DB gdb
}

In [22]:
# Define map area
map_area = gpd.read_file(CONFIG[CONFIG_MAP_AREA_PATH]).to_crs(GEODESIC_CRS).loc[0,"geometry"]
map_area

DataSourceError: ./rawData/santacruz_county.geojson: No such file or directory

In [ ]:
# Instantiate data objects

gtfs_instance = GTFSDataObject(
    CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS],
    CONFIG[CONFIG_GTFS_CACHE_FOLDER],
    CONFIG[CONFIG_GTFS_TRANSITLAND_URL],
    api_key_path=CONFIG[CONFIG_GTFS_TRANSITLAND_KEY_PATH],
)
fta_instance = FTAFacilityInventoryDataObject(
    CONFIG[CONFIG_FTA_FACILITY_INVENTORY_PATH],
    CONFIG[CONFIG_OSM_CACHE_FOLDER]
)
citybikes_instance = CityBikesDataObject(CONFIG[CONFIG_CITYBIKES_URL])
afdc_instance = AFDCApiDataObject(CONFIG[CONFIG_AFDC_URL], CONFIG[CONFIG_AFDC_API_KEY],CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
osm_bike_parking_instance = OSMBikeParkingDataObject(CONFIG[CONFIG_OSM_CACHE_FOLDER], {"amenity": ["bicycle_parking"]})
smart_location_info = SmartLocationWrapper(CONFIG[CONFIG_SMART_LOCATION_PATH], CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
base_layer_instance = BaseLayer(
    [
        SmartLocationPopulationDensity(smart_location_info),
        SmartLocationJobDensity(smart_location_info),
        SmartLocationRetailEntertainmentJobDensity(smart_location_info),
        SmartLocationRawJobs(smart_location_info),
        CensusModeshare(),
        CensusCarlessness(),
        SmartLocationNationalWalkabilityIndex(smart_location_info),
        BaseLayerEjscreen(CONFIG[CONFIG_EPA_EJSCREEN_PATH]),
    ],
    CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS],
    ColorMaps.NATIONAL_WALKABILITY_INDEX_COLORMAP,
    smooth=True,
    remove_water=True # This should be set to False if TIGER goes down
)
bike_instance = OSMBikeStreetsDataObject(
    CONFIG[CONFIG_OSM_CACHE_FOLDER],
    reference=gtfs_instance,
    local_crs=CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS]
)

# Edit these columns to change the objects displayed on the map
all_objects = {
    "GTFS": gtfs_instance,
    "BASE": base_layer_instance,
    "BIKE": bike_instance,
    "FTA": fta_instance,
    "CITYBIKES": citybikes_instance,
    "AFDC": afdc_instance,
    "OSM BIKE PARKING": osm_bike_parking_instance,
}
objects_load_order = (
    "BASE",
    "BIKE",
    "GTFS",
    "FTA",
    "AFDC",
    "OSM BIKE PARKING",
    "CITYBIKES"
)

# This should only contain GTFS
objects_must_await = (
    "GTFS"
)
await gtfs_instance.load_data(map_area, GEODESIC_CRS)
base_layer_instance.load_data(map_area, GEODESIC_CRS)

#### Mobility Hub Map

In [ ]:
# Mobility Hub Only Map
mobility_hub_instance = MobilityHubDataObject(gtfs_instance, base_layer_instance, CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
mobility_hub_instance.load_data(map_area, GEODESIC_CRS)
mobility_hub_map = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="Cartodb Positron",
    zoom_start=10,
    prefer_canvas=True
)
base_layer_instance.get_folium_plot().add_to(mobility_hub_map)
mobility_hub_instance.get_folium_plot().add_to(mobility_hub_map)
mobility_hub_map

#### All Elements

In [ ]:
# Load data objects
for name, object in all_objects.items():
    print(name)
    if not object.get_is_loaded():
        if name in objects_must_await:
            await object.load_data(map_area, GEODESIC_CRS)
        else:
            object.load_data(map_area, GEODESIC_CRS)

In [ ]:
# Main Map
display_map = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="Cartodb Positron",
    zoom_start=10,
    prefer_canvas=True
)
for object in objects_load_order:
    print(object)
    assert all_objects[object].get_is_loaded()
    if len(all_objects[object].gdf) > 0: #TODO: quick fix, add "has_entries" method to each data object instead
        all_objects[object].get_folium_plot().add_to(display_map)
#all_objects["GTFS"].get_folium_plot().add_to(display_map)
display_map

### Example Export Function

In [ ]:
import pathlib
def export_data_object_as_layer(data_object_instance, output_folder):
    """Export data_object_instance to output_folder as a geojson"""
    gdf = data_object_instance.gdf
    gdf.to_file(pathlib.Path(output_folder) / f"{data_object_instance.name}.geojson")

objects_to_export = [gtfs_instance, mobility_hub_instance, citybikes_instance, base_layer_instance, afdc_instance, fta_instance]
for object in objects_to_export:
    print(object.name)
    export_data_object_as_layer(object, "test")
